In [1]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import seaborn as sns
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
import time

%matplotlib inline

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 150)
sns.set_style('darkgrid')
matplotlib.rcParams['font.size'] = 14
matplotlib.rcParams['figure.figsize'] = (10, 6)
matplotlib.rcParams['figure.facecolor'] = '#00000000'

In [2]:
from preprocessing.preprocess_numeric import preprocess_numeric
from preprocessing.preprocess_tfidf import preprocess_tfidf
from preprocessing.preprocess_st import clean_st
from sentence_transformers import SentenceTransformer
import numpy as np

In [3]:
# Load dataset
df = pd.read_csv("data/Data/News_Final.csv")
df.set_index("IDLink", inplace=True)
df = df.copy()

In [4]:
# Numeric features
df_processed, X_num, num_cols, scaler = preprocess_numeric(df)

In [5]:
# TF-IDF
X_tfidf, tfidf_title, tfidf_headline, tfidf_source = preprocess_tfidf(df_processed)

In [6]:
df_processed.columns

Index(['Title', 'Headline', 'Source', 'PublishDate', 'SentimentTitle',
       'SentimentHeadline', 'Facebook', 'GooglePlus', 'LinkedIn', 'year',
       'month', 'day', 'weekday', 'hour', 'Topic_microsoft', 'Topic_obama',
       'Topic_palestine', 'title_length_chars', 'title_length_words',
       'headline_length_chars', 'headline_length_words',
       'headline_punctuation_count', 'headline_uppercase_ratio',
       'engagement_total', 'is_viral'],
      dtype='object')

In [7]:
# Sentence-Transformers embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')
emb_title = model.encode(df_processed['Title'].apply(clean_st).tolist())
emb_headline = model.encode(df_processed['Headline'].apply(clean_st).tolist())
emb_source = model.encode(df_processed['Source'].apply(clean_st).tolist())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
np.save("emb_title.npy", emb_title)
np.save("emb_headline.npy", emb_headline)
np.save("emb_source.npy", emb_source)

In [9]:
emb_title = np.load("emb_title.npy")
emb_headline = np.load("emb_headline.npy")
emb_source = np.load("emb_source.npy")

In [10]:
X_emb = np.hstack([emb_title, emb_headline, emb_source])

In [11]:
from scipy.sparse import csr_matrix, hstack

In [12]:
X_final_tfidf = hstack([X_tfidf, X_num])

In [14]:
X_emb_sparse = csr_matrix(X_emb)
X_num_sparse = csr_matrix(X_num)

In [15]:
X_final_emb = hstack([X_emb_sparse, X_num_sparse])

In [18]:
from sklearn.model_selection import train_test_split

idx = np.arange(len(df))

train_idx, temp_idx = train_test_split(idx, test_size=0.30, random_state=42)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, random_state=42)

In [19]:
np.save("train_idx.npy", train_idx)
np.save("val_idx.npy", val_idx)
np.save("test_idx.npy", test_idx)

In [20]:
train_idx = np.load("train_idx.npy")
val_idx = np.load("val_idx.npy")
test_idx = np.load("test_idx.npy")

In [21]:
# convert to CSR for slicing
X_final_tfidf = csr_matrix(X_final_tfidf)

In [22]:
X_tfidf_train = X_final_tfidf[train_idx]
X_tfidf_val   = X_final_tfidf[val_idx]
X_tfidf_test  = X_final_tfidf[test_idx]

In [23]:
# Ensure CSR format for slicing
X_final_emb = csr_matrix(X_final_emb)

In [24]:
X_emb_train = X_final_emb[train_idx]
X_emb_val   = X_final_emb[val_idx]
X_emb_test  = X_final_emb[test_idx]

In [25]:
df_processed["bert_text"] = (
    df_processed["Title"].fillna("") + " [SEP] " +
    df_processed["Headline"].fillna("") + " [SEP] " +
    df_processed["Source"].fillna("")
)

In [26]:
X_text = df_processed["bert_text"].values

In [27]:
X_text_train, X_text_val, X_text_test = X_text[train_idx], X_text[val_idx], X_text[test_idx]
X_num_train, X_num_val, X_num_test = X_num[train_idx], X_num[val_idx], X_num[test_idx]

In [28]:
df_processed.info()

<class 'pandas.core.frame.DataFrame'>
Index: 93239 entries, 99248.0 to 61870.0
Data columns (total 26 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   Title                       93239 non-null  object        
 1   Headline                    93239 non-null  object        
 2   Source                      93239 non-null  object        
 3   PublishDate                 93239 non-null  datetime64[ns]
 4   SentimentTitle              93239 non-null  float64       
 5   SentimentHeadline           93239 non-null  float64       
 6   Facebook                    93239 non-null  int64         
 7   GooglePlus                  93239 non-null  int64         
 8   LinkedIn                    93239 non-null  int64         
 9   year                        93239 non-null  int32         
 10  month                       93239 non-null  int32         
 11  day                         93239 non-null  int32  

In [29]:
df_processed['Facebook_log'] = np.log1p(df_processed['Facebook'])
df_processed['GooglePlus_log'] = np.log1p(df_processed['GooglePlus'])
df_processed['LinkedIn_log'] = np.log1p(df_processed['LinkedIn'])

In [30]:
y_fb = df_processed["Facebook_log"].values
y_gp = df_processed["GooglePlus_log"].values
y_li = df_processed["LinkedIn_log"].values

In [33]:
y_viral = df_processed["is_viral"].values

In [34]:
y_fb_train = y_fb[train_idx]
y_fb_val   = y_fb[val_idx]
y_fb_test  = y_fb[test_idx]

y_gp_train = y_gp[train_idx]
y_gp_val   = y_gp[val_idx]
y_gp_test  = y_gp[test_idx]

y_li_train = y_li[train_idx]
y_li_val   = y_li[val_idx]
y_li_test  = y_li[test_idx]

In [35]:
y_viral_train = y_viral[train_idx]
y_viral_val   = y_viral[val_idx]
y_viral_test  = y_viral[test_idx]

In [36]:
from scipy import sparse

In [37]:
sparse.save_npz("X_tfidf_train.npz", X_tfidf_train)
sparse.save_npz("X_tfidf_val.npz", X_tfidf_val)
sparse.save_npz("X_tfidf_test.npz", X_tfidf_test)

In [38]:
sparse.save_npz("X_emb_train.npz", X_emb_train)
sparse.save_npz("X_emb_val.npz", X_emb_val)
sparse.save_npz("X_emb_test.npz", X_emb_test)

In [39]:
np.save("X_text_train.npy", X_text_train)
np.save("X_text_val", X_text_val)
np.save("X_text_test.npy", X_text_test)

np.save("X_num_train.npy", X_num_train)
np.save("X_num_val", X_num_val)
np.save("X_num_test.npy", X_num_test)

In [40]:
np.save("y_fb_train.npy", y_fb_train)
np.save("y_fb_val.npy", y_fb_val)
np.save("y_fb_test.npy", y_fb_test)

np.save("y_gp_train.npy", y_gp_train)
np.save("y_gp_val.npy", y_gp_val)
np.save("y_gp_test.npy", y_gp_test)

np.save("y_li_train.npy", y_li_train)
np.save("y_li_val.npy", y_li_val)
np.save("y_li_test.npy", y_li_test)

In [41]:
np.save("y_viral_train.npy", y_viral_train)
np.save("y_viral_val.npy", y_viral_val)
np.save("y_viral_test.npy", y_viral_test)

In [42]:
X_tfidf_train = sparse.load_npz("X_tfidf_train.npz")
X_tfidf_val   = sparse.load_npz("X_tfidf_val.npz")
X_tfidf_test  = sparse.load_npz("X_tfidf_test.npz")

In [43]:
X_emb_train = sparse.load_npz("X_emb_train.npz")
X_emb_val = sparse.load_npz("X_emb_val.npz")
X_emb_test = sparse.load_npz("X_emb_test.npz")

In [44]:
y_viral_train = np.load("y_viral_train.npy")
y_viral_val = np.load("y_viral_val.npy")
y_viral_test = np.load("y_viral_test.npy")

In [45]:
X_text_train = np.load("X_text_train.npy", allow_pickle=True)
X_text_val = np.load("X_text_val.npy", allow_pickle=True)
X_text_test = np.load("X_text_test.npy", allow_pickle=True)

X_num_train = np.load("X_num_train.npy", allow_pickle=True)
X_num_val = np.load("X_num_val.npy", allow_pickle=True)
X_num_test = np.load("X_num_test.npy", allow_pickle=True)

In [46]:
datasets = {
    "train_idx": train_idx,
    "val_idx": val_idx,
    "test_idx": test_idx,

    "X_tfidf_train": X_tfidf_train,
    "X_tfidf_val": X_tfidf_val,
    "X_tfidf_test": X_tfidf_test,

    "X_emb_train": X_emb_train,
    "X_emb_val": X_emb_val,
    "X_emb_test": X_emb_test,

    "X_num_train": X_num_train,
    "X_num_val": X_num_val,
    "X_num_test": X_num_test,

    "X_text_train": X_text_train,
    "X_text_val": X_text_val,
    "X_text_test": X_text_test,

    "y_fb_train": y_fb_train,
    "y_fb_val": y_fb_val,
    "y_fb_test": y_fb_test,

    "y_gp_train": y_gp_train,
    "y_gp_val": y_gp_val,
    "y_gp_test": y_gp_test,

    "y_li_train": y_li_train,
    "y_li_val": y_li_val,
    "y_li_test": y_li_test,

    "y_viral_train": y_viral_train,
    "y_viral_val": y_viral_val,
    "y_viral_test": y_viral_test,
}

for name, arr in datasets.items():
    try:
        length = arr.shape[0]
    except:
        length = len(arr)
    print(f"{name:20s} → {length}")

train_idx            → 65267
val_idx              → 13986
test_idx             → 13986
X_tfidf_train        → 65267
X_tfidf_val          → 13986
X_tfidf_test         → 13986
X_emb_train          → 65267
X_emb_val            → 13986
X_emb_test           → 13986
X_num_train          → 65267
X_num_val            → 13986
X_num_test           → 13986
X_text_train         → 65267
X_text_val           → 13986
X_text_test          → 13986
y_fb_train           → 65267
y_fb_val             → 13986
y_fb_test            → 13986
y_gp_train           → 65267
y_gp_val             → 13986
y_gp_test            → 13986
y_li_train           → 65267
y_li_val             → 13986
y_li_test            → 13986
y_viral_train        → 65267
y_viral_val          → 13986
y_viral_test         → 13986


In [47]:
df_processed.columns

Index(['Title', 'Headline', 'Source', 'PublishDate', 'SentimentTitle',
       'SentimentHeadline', 'Facebook', 'GooglePlus', 'LinkedIn', 'year',
       'month', 'day', 'weekday', 'hour', 'Topic_microsoft', 'Topic_obama',
       'Topic_palestine', 'title_length_chars', 'title_length_words',
       'headline_length_chars', 'headline_length_words',
       'headline_punctuation_count', 'headline_uppercase_ratio',
       'engagement_total', 'is_viral', 'bert_text', 'Facebook_log',
       'GooglePlus_log', 'LinkedIn_log'],
      dtype='object')

In [48]:
df_processed[df_processed["Facebook"] > 1].describe()

,PublishDate,SentimentTitle,SentimentHeadline,Facebook,GooglePlus,LinkedIn,year,month,day,weekday,hour,title_length_chars,title_length_words,headline_length_chars,headline_length_words,headline_punctuation_count,headline_uppercase_ratio,engagement_total,is_viral,Facebook_log,GooglePlus_log,LinkedIn_log
count,58380,58380.000000,58380.000000,58380.000000,58380.00000,58380.000000,58380.000000,58380.000000,58380.000000,58380.000000,58380.000000,58380.000000,58380.000000,58380.00000,58380.000000,58380.000000,58380.000000,58380.000000,58380.000000,58380.000000,58380.000000,58380.000000
mean,2016-03-01 10:01:33.534481152,-0.006410,-0.025879,180.787496,5.95322,21.221377,2015.776276,5.184156,15.528075,2.525762,12.638541,57.670007,9.220469,152.62554,24.672405,1.940870,0.046854,207.962093,0.156800,3.342199,0.949348,1.280905
min,2015-11-08 09:15:00,-0.950694,-0.755355,2.000000,0.00000,0.000000,2015.000000,1.000000,1.000000,0.000000,0.000000,4.000000,1.000000,0.00000,0.000000,0.000000,0.000000,2.000000,0.000000,1.098612,0.000000,0.000000
25%,2016-01-06 20:24:44,-0.079057,-0.111803,6.000000,0.00000,0.000000,2016.000000,2.000000,8.000000,1.000000,7.000000,51.000000,8.000000,135.00000,21.000000,1.000000,0.025641,9.000000,0.000000,1.945910,0.000000,0.000000
50%,2016-02-28 14:41:25.500000,0.000000,-0.025000,20.000000,1.00000,1.000000,2016.000000,4.000000,16.000000,2.000000,14.000000,60.000000,9.000000,139.00000,23.000000,2.000000,0.040541,30.000000,0.000000,3.044522,0.693147,0.693147
75%,2016-04-25 23:41:06,0.062500,0.061286,85.000000,4.00000,9.000000,2016.000000,6.000000,23.000000,4.000000,18.000000,66.000000,11.000000,144.00000,25.000000,3.000000,0.060092,119.000000,0.000000,4.454347,1.609438,2.302585
max,2016-07-07 15:38:26,0.962354,0.964646,49211.000000,1267.00000,6362.000000,2016.000000,12.000000,31.000000,6.000000,23.000000,170.000000,24.000000,446.00000,79.000000,66.000000,0.790909,49211.000000,1.000000,10.803893,7.145196,8.758255
std,NaN,0.136713,0.141614,775.908582,22.49262,97.350920,0.416743,3.772462,8.786800,1.868803,6.804822,12.013678,2.243488,54.20338,9.003559,1.625289,0.031583,806.275904,0.363615,1.722330,1.136627,1.580880


In [49]:
df_processed[df_processed["is_viral"] == 1].describe()

,PublishDate,SentimentTitle,SentimentHeadline,Facebook,GooglePlus,LinkedIn,year,month,day,weekday,hour,title_length_chars,title_length_words,headline_length_chars,headline_length_words,headline_punctuation_count,headline_uppercase_ratio,engagement_total,is_viral,Facebook_log,GooglePlus_log,LinkedIn_log
count,9331,9331.000000,9331.000000,9331.000000,9331.000000,9331.000000,9331.000000,9331.000000,9331.000000,9331.000000,9331.000000,9331.000000,9331.000000,9331.000000,9331.000000,9331.000000,9331.000000,9331.000000,9331.0,9331.000000,9331.000000,9331.000000
mean,2016-02-28 10:52:05.735076864,-0.009870,-0.028010,956.352481,25.587075,112.790376,2015.759511,5.321295,15.485693,2.591148,12.891866,58.265031,9.463402,146.450005,23.657164,1.826492,0.047858,1094.729932,1.0,6.107371,2.382657,2.601325
min,2015-11-09 00:00:00,-0.632456,-0.689588,0.000000,0.000000,0.000000,2015.000000,1.000000,1.000000,0.000000,0.000000,9.000000,1.000000,11.000000,2.000000,0.000000,0.000000,241.000000,1.0,0.000000,0.000000,0.000000
25%,2016-01-03 21:47:31.500000,-0.083333,-0.112268,270.000000,3.000000,1.000000,2016.000000,2.000000,8.000000,1.000000,8.000000,52.000000,8.000000,135.000000,21.000000,1.000000,0.027778,351.000000,1.0,5.602119,1.386294,0.693147
50%,2016-02-19 14:15:03,0.000000,-0.026064,450.000000,10.000000,9.000000,2016.000000,4.000000,15.000000,2.000000,14.000000,61.000000,10.000000,139.000000,23.000000,2.000000,0.041958,552.000000,1.0,6.111467,2.397895,2.302585
75%,2016-04-26 07:38:07,0.055693,0.057197,954.500000,27.000000,77.000000,2016.000000,7.000000,23.000000,4.000000,18.000000,66.000000,11.000000,143.000000,24.000000,3.000000,0.061069,1092.500000,1.0,6.862235,3.332205,4.356709
max,2016-07-07 12:22:05,0.617945,0.621255,49211.000000,1267.000000,20341.000000,2016.000000,12.000000,31.000000,6.000000,23.000000,170.000000,22.000000,429.000000,78.000000,14.000000,0.343511,49211.000000,1.0,10.803893,7.145196,9.920443
std,NaN,0.136091,0.140919,1743.658683,51.402426,474.366988,0.427403,3.891298,8.649213,1.861128,6.974244,11.993369,2.275134,40.994635,6.946498,1.524227,0.029846,1816.959400,0.0,1.411722,1.363964,2.155369
